In [0]:
from datetime import datetime
df = spark.read.format('json')\
    .load(f'abfss://raw@datalakestrgdev.dfs.core.windows.net/airlines/{datetime.now().strftime('year=%Y/month=%m/day=%d')}/')

In [0]:
from pyspark.sql.functions import explode, current_timestamp

df1 = df.select(explode('response'))
df_final = df1.select('col.*')\
              .withColumn('last_load_ts', current_timestamp())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, col, lit

silver_table = "projectdev.cleansed.airlines"

if not spark.catalog.tableExists(silver_table):
    df_final.write.format('delta').mode('overwrite').saveAsTable(silver_table)
else:
    last_watermark = spark.sql(f"""
        SELECT coalesce(max(last_load_ts), timestamp('1900-01-01'))
        FROM {silver_table}
    """).collect()[0][0]

    incremental_df = df_final.filter(col("last_load_ts") > lit(last_watermark))\
                                .dropDuplicates(["iata_code"])

    delta_table = DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
        .merge(
            incremental_df.alias("s"),
            "t.iata_code = s.iata_code"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# df_final.write.format('delta')\
#     .option("mergeSchema", "true")\
#     .mode('overwrite').save('abfss://cleansed@datalakestrgdev.dfs.core.windows.net/airlines')

In [0]:
%sql
-- for external delta files
create table if not exists projectdev.cleansed.airlines
using delta
location 'abfss://cleansed@datalakestrgdev.dfs.core.windows.net/airlines'